# Self-Healing Governance — Dev Log

## Objetivo

`check_and_heal()` detecta falha de um check de saúde já computado por
outro módulo, abre um incidente REAL via `incident_response`, e sugere
remediação de um catálogo declarativo. **Nunca corrige dados/código
sozinho** — "self-healing" aqui é detectar+registrar+recomendar, por
decisão de risco deliberada.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path
from core.audit_logs.logger import AuditLogger
from core.blockchain_audit_layer.engine import create_checkpoint, verify_checkpoint_chain
from core.incident_response.log import IncidentLog
from core.self_healing_governance.healer import check_and_heal
from shared.schemas import AuditEventType

demo_dir = Path(tempfile.mkdtemp(prefix="healing_demo_"))
audit_logger = AuditLogger(log_path=demo_dir / "audit_log.jsonl")
checkpoint_path = demo_dir / "checkpoints.jsonl"
incident_log = IncidentLog(storage_path=demo_dir / "incidents.json")

audit_logger.record_event(AuditEventType.PII_SCAN, actor="demo", payload={})
create_checkpoint(logger=audit_logger, checkpoint_path=checkpoint_path)
healthy = verify_checkpoint_chain(checkpoint_path=checkpoint_path)
actions = check_and_heal({"checkpoint_chain_integrity": healthy}, incident_log=incident_log)
print(f"Cadeia íntegra: {healthy} -> ação: healthy={actions[0].healthy}, incidente={actions[0].incident_id}")

import json as _json
lines = checkpoint_path.read_text(encoding="utf-8").splitlines()
record = _json.loads(lines[0])
record["merkle_root"] = "0" * 64
checkpoint_path.write_text(_json.dumps(record) + "\n", encoding="utf-8")

tampered = verify_checkpoint_chain(checkpoint_path=checkpoint_path)
actions2 = check_and_heal({"checkpoint_chain_integrity": tampered}, incident_log=incident_log)
print(f"Cadeia adulterada: {tampered} -> ação: healthy={actions2[0].healthy}, incidente={actions2[0].incident_id[:8]}...")
print("Passos sugeridos:")
for step in actions2[0].suggested_steps:
    print(f"  - {step}")

Cadeia íntegra: True -> ação: healthy=True, incidente=None
Cadeia adulterada: False -> ação: healthy=False, incidente=3fb3616e...
Passos sugeridos:
  - Mesma investigação de audit_chain_integrity, aplicada à cadeia de checkpoints Merkle.
  - Verificar se algum merkle_root publicado externamente (se houver) diverge do local.


Integração real de ponta a ponta: `blockchain_audit_layer` detecta a
adulteração de verdade (não simulada), e `self_healing_governance` abre um
incidente real em resposta — nenhum dos dois lados é mockado.

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/self_healing_governance/tests -v
```

5/5 testes passando.